# 01 · Mine related-work sections and citation contexts from arXiv

**Goal.** Build the datasets every later stage depends on:

| File | What it holds | Used by |
|---|---|---|
| `citation_contexts.jsonl` | one row per (citing paper, cited paper, sentence that cites it) | relation classifier (A) |
| `rw_pairs.jsonl` | a paper's related-work section + titles/abstracts of the papers it cites | evaluation benchmark (#6), generator (C) |
| `openalex_cache.json` | cited-title → OpenAlex metadata | everything |

**Time split.** Train papers come from an older window, test papers from a recent one. Test papers should be newer than the training cutoff of any LLM you evaluate, otherwise the model may have seen the real related-work section. Adjust `TEST_WINDOW` once you know which models you'll use.

Everything is cached to Google Drive and every stage is **resumable**: if Colab disconnects, re-run all cells and it continues where it stopped. No GPU needed for this notebook.


In [ ]:
!pip -q install pylatexenc rapidfuzz feedparser

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
ROOT = Path('/content/drive/MyDrive/litreview')
for sub in ['data', 'sources']:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
DATA, SRC = ROOT / 'data', ROOT / 'sources'
print('Working in', ROOT)

### Parsing library
Pure functions, tested offline: finds the main `.tex`, inlines `\input{}`, strips comments, locates the related-work section, converts LaTeX to text while keeping citations as `[CITE:key]` markers, and maps cite keys to titles from `.bbl`/`.bib`.

In [ ]:
%%writefile /content/latex_mining.py
"""Core LaTeX mining logic: related-work sections, citation contexts, bibliography keys.

Pure functions, no network access, so they can be unit-tested offline.
"""
import re
from pathlib import Path

from pylatexenc.latex2text import LatexNodes2Text

CITE_RE = re.compile(
    r"\\(?:cite|citep|citet|citealp|citealt|citeauthor|citeyear|parencite|textcite|autocite)\*?"
    r"(?:\[[^\]]*\]){0,2}\{([^}]+)\}"
)
SECTION_RE = re.compile(r"\\(section|subsection)\*?\{([^}]*)\}")
RW_TITLES = re.compile(
    r"related\s*work|prior\s*work|previous\s*work|background|literature\s*review|related\s*literature",
    re.I,
)
INPUT_RE = re.compile(r"\\(?:input|include)\{([^}]+)\}")


def strip_comments(tex: str) -> str:
    """Remove LaTeX comments (unescaped %) line by line."""
    return re.sub(r"(?<!\\)%.*", "", tex)


def find_main_tex(files: dict) -> str | None:
    """Pick the .tex file that has \\documentclass and the most content."""
    cands = [n for n, t in files.items() if n.endswith(".tex") and "\\documentclass" in t]
    if not cands:
        return None
    return max(cands, key=lambda n: len(files[n]))


def inline_inputs(tex: str, files: dict, depth: int = 0) -> str:
    """Recursively replace \\input{x} / \\include{x} with file contents."""
    if depth > 5:
        return tex

    def repl(m):
        name = m.group(1).strip()
        for cand in (name, name + ".tex"):
            for fname, content in files.items():
                if fname == cand or fname.endswith("/" + cand):
                    return inline_inputs(strip_comments(content), files, depth + 1)
        return ""

    return INPUT_RE.sub(repl, tex)


def extract_sections(tex: str) -> list[dict]:
    """Split body into top-level sections: [{title, body}]."""
    body = tex.split("\\begin{document}", 1)[-1].split("\\end{document}", 1)[0]
    marks = [(m.start(), m.end(), m.group(1), m.group(2)) for m in SECTION_RE.finditer(body)]
    tops = [m for m in marks if m[2] == "section"]
    out = []
    for i, (s, e, _, title) in enumerate(tops):
        end = tops[i + 1][0] if i + 1 < len(tops) else len(body)
        out.append({"title": title.strip(), "body": body[e:end]})
    return out


def find_related_work(tex: str) -> dict | None:
    """Return the first section whose title looks like related work."""
    for sec in extract_sections(tex):
        if RW_TITLES.search(sec["title"]):
            return sec
    return None


def _cite_to_placeholder(tex: str) -> str:
    """Replace citation commands with [CITE:k1,k2] tokens that survive LaTeX->text."""
    return CITE_RE.sub(lambda m: " CITEOPEN" + m.group(1).replace(" ", "") + "CITECLOSE ", tex)


_l2t = LatexNodes2Text(math_mode="verbatim")


def latex_to_text(tex: str) -> str:
    """Convert LaTeX to plain text while keeping citation markers as [CITE:keys]."""
    tex = re.sub(r"\\begin\{(figure|table|algorithm|equation)\*?\}.*?\\end\{\1\*?\}", " ", tex, flags=re.S)
    tex = _cite_to_placeholder(tex)
    try:
        text = _l2t.latex_to_text(tex)
    except Exception:
        text = re.sub(r"\\[a-zA-Z]+\*?(\[[^\]]*\])?", " ", tex)
    text = re.sub(r"CITEOPEN(.*?)CITECLOSE", lambda m: f"[CITE:{m.group(1)}]", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" ([.,;:)])", r"\1", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z\[])")


def split_sentences(text: str) -> list[str]:
    text = re.sub(r"(e\.g|i\.e|et al|cf|vs|Fig|Sec|Eq)\.", lambda m: m.group(1) + "<DOT>", text)
    sents = [s.replace("<DOT>", ".").strip() for s in SENT_SPLIT.split(text)]
    return [s for s in sents if len(s) > 20]


def citation_contexts(text: str, window: int = 1) -> list[dict]:
    """For every sentence with a [CITE:...] marker, emit one record per cited key.

    `context` includes `window` sentences either side; `cite_sentence` is the sentence itself.
    """
    sents = split_sentences(text)
    out = []
    for i, s in enumerate(sents):
        keys = re.findall(r"\[CITE:([^\]]+)\]", s)
        if not keys:
            continue
        ctx = " ".join(sents[max(0, i - window): i + window + 1])
        all_keys = [k for group in keys for k in group.split(",") if k]
        for k in all_keys:
            out.append({
                "cite_key": k,
                "cite_sentence": s,
                "context": ctx,
                "n_cites_in_sentence": len(all_keys),
            })
    return out


def paragraphs(text: str) -> list[str]:
    return [p.strip() for p in re.split(r"\n\s*\n", text) if len(p.strip()) > 80]


# ---------- bibliography ----------

def parse_bbl(bbl: str) -> dict:
    """Map cite key -> rough title from a compiled .bbl (natbib / plain styles)."""
    out = {}
    for m in re.finditer(r"\\bibitem(?:\[[^\]]*\])?\{([^}]+)\}(.*?)(?=\\bibitem|\\end\{thebibliography\})", bbl, re.S):
        key, entry = m.group(1).strip(), m.group(2)
        blocks = [b.strip() for b in re.split(r"\\newblock", entry)]
        title = blocks[1] if len(blocks) > 1 else blocks[0]
        title = re.sub(r"[{}\\]|\s+", " ", title)
        title = re.sub(r"\b(emph|em|textit|bf)\b", "", title).strip(" .,")
        out[key] = title
    return out


def parse_bib(bib: str) -> dict:
    """Map cite key -> title from a .bib file."""
    out = {}
    for m in re.finditer(r"@\w+\s*\{\s*([^,\s]+)\s*,(.*?)(?=\n@|\Z)", bib, re.S):
        key, body = m.group(1), m.group(2)
        t = re.search(r"\btitle\s*=\s*[{\"](.+?)[}\"]\s*,?\s*\n", body, re.S | re.I)
        if t:
            out[key] = re.sub(r"[{}]|\s+", " ", t.group(1)).strip()
    return out


def build_keymap(files: dict) -> dict:
    keymap = {}
    for name, content in files.items():
        if name.endswith(".bib"):
            keymap.update(parse_bib(content))
    for name, content in files.items():  # .bbl wins: it's what was actually compiled
        if name.endswith(".bbl"):
            keymap.update(parse_bbl(content))
    return keymap


def mine_paper(files: dict) -> dict | None:
    """files: {relative_path: text}. Returns related-work text, contexts, and key->title map."""
    main = find_main_tex(files)
    if not main:
        return None
    tex = inline_inputs(strip_comments(files[main]), files)
    rw = find_related_work(tex)
    keymap = build_keymap(files)
    full_text = latex_to_text(tex.split("\\begin{document}", 1)[-1])
    result = {
        "main_file": main,
        "keymap": keymap,
        "all_contexts": citation_contexts(full_text),
        "related_work": None,
    }
    if rw:
        rw_text = latex_to_text(rw["body"])
        result["related_work"] = {
            "title": rw["title"],
            "text": rw_text,
            "paragraphs": paragraphs(rw_text),
            "cited_keys": sorted({k for c in citation_contexts(rw_text) for k in [c["cite_key"]]}),
        }
    return result


In [ ]:
import sys, importlib
sys.path.insert(0, '/content')
import latex_mining as lm
importlib.reload(lm)

# quick self-test on a tiny synthetic paper
_test = lm.mine_paper({
    'main.tex': r'\documentclass{article}\begin{document}\section{Related Work}'
                r'Pointer networks were used by \citet{see2017}. In contrast, \citep{liu2019} fine-tune BERT encoders for summarization.'
                r'\section{Method}x\end{document}',
    'refs.bib': '@article{see2017,\n title = {Get to the point},\n}\n',
})
assert _test['related_work']['cited_keys'] == ['liu2019', 'see2017'], _test
print('parser OK:', _test['related_work']['text'])

### Config
Start with a small pilot (a few hundred papers) to check yield, then scale up. Roughly 50–60% of arXiv papers have usable LaTeX with an identifiable related-work section.

In [ ]:
CATEGORIES   = ['cs.CL', 'cs.IR']            # NLP + IR: closest to the project's domain
TRAIN_WINDOW = ('202301010000', '202412312359')
TEST_WINDOW  = ('202603010000', '202609302359')  # keep newer than your eval models' cutoffs
N_TRAIN, N_TEST = 400, 120                      # pilot sizes; scale to ~3000 / 300 later
OPENALEX_MAILTO = 'you@example.com'             # put your email: gets the faster 'polite pool'
ARXIV_DELAY = 3.1                               # arXiv asks for >= 3s between requests

### 1. List papers per split (arXiv API)

In [ ]:
import time, json, feedparser, requests, random

def list_arxiv(window, n, cats=CATEGORIES, page=200):
    cat_q = ' OR '.join(f'cat:{c}' for c in cats)
    q = f'({cat_q}) AND submittedDate:[{window[0]} TO {window[1]}]'
    out, start = [], 0
    while len(out) < n:
        r = requests.get('http://export.arxiv.org/api/query', params={
            'search_query': q, 'start': start, 'max_results': page,
            'sortBy': 'submittedDate', 'sortOrder': 'descending'}, timeout=60)
        feed = feedparser.parse(r.text)
        if not feed.entries:
            break
        for e in feed.entries:
            out.append({'arxiv_id': e.id.split('/abs/')[-1].split('v')[0],
                        'title': ' '.join(e.title.split()),
                        'abstract': ' '.join(e.summary.split()),
                        'published': e.published[:10],
                        'primary_cat': e.arxiv_primary_category['term']})
        start += page
        time.sleep(ARXIV_DELAY)
    return out

def load_or_list(split, window, n):
    path = DATA / f'papers_{split}.jsonl'
    if path.exists():
        return [json.loads(l) for l in path.open()]
    # over-fetch, then sample, so the split isn't just the last few days of the window
    papers = list_arxiv(window, n * 4)
    random.Random(0).shuffle(papers)
    papers = papers[:n]
    with path.open('w') as f:
        for p in papers: f.write(json.dumps(p) + '\n')
    return papers

papers = {'train': load_or_list('train', TRAIN_WINDOW, N_TRAIN),
          'test':  load_or_list('test',  TEST_WINDOW,  N_TEST)}
for s, ps in papers.items():
    print(s, len(ps), 'papers,', min(p['published'] for p in ps), '→', max(p['published'] for p in ps))

### 2. Download LaTeX sources
Cached per paper on Drive. PDF-only submissions are skipped.

In [ ]:
import tarfile, gzip, io
from tqdm.auto import tqdm

KEEP_EXT = ('.tex', '.bbl', '.bib')
MAX_FILE = 2_000_000

def fetch_source(arxiv_id):
    """Return {relative_path: text} for the paper's LaTeX files, or None."""
    cache = SRC / f"{arxiv_id.replace('/', '_')}.json"
    if cache.exists():
        return json.loads(cache.read_text()) or None
    r = requests.get(f'https://arxiv.org/e-print/{arxiv_id}', timeout=120,
                     headers={'User-Agent': 'litreview-research (student project)'})
    time.sleep(ARXIV_DELAY)
    files, blob = {}, r.content
    if r.ok and not blob.startswith(b'%PDF'):
        try:
            with tarfile.open(fileobj=io.BytesIO(blob), mode='r:*') as tar:
                for m in tar.getmembers():
                    if m.isfile() and m.name.lower().endswith(KEEP_EXT) and m.size < MAX_FILE:
                        files[m.name] = tar.extractfile(m).read().decode('utf-8', 'ignore')
        except tarfile.ReadError:
            try:  # single gzipped .tex file
                files['main.tex'] = gzip.decompress(blob).decode('utf-8', 'ignore')
            except OSError:
                pass
    cache.write_text(json.dumps(files))   # {} marks 'no usable source', so we don't retry
    return files or None

for split, ps in papers.items():
    ok = sum(fetch_source(p['arxiv_id']) is not None for p in tqdm(ps, desc=split))
    print(f'{split}: {ok}/{len(ps)} with LaTeX source')

### 3. Parse every paper

In [ ]:
mined = {}
fail = {'no_source': 0, 'no_main': 0, 'no_rw': 0, 'error': 0}
for split, ps in papers.items():
    for p in tqdm(ps, desc=split):
        files = fetch_source(p['arxiv_id'])
        if not files:
            fail['no_source'] += 1; continue
        try:
            res = lm.mine_paper(files)
        except Exception as e:
            fail['error'] += 1; continue
        if res is None:
            fail['no_main'] += 1; continue
        if res['related_work'] is None:
            fail['no_rw'] += 1
        mined[p['arxiv_id']] = {**p, 'split': split, **res}

n_rw = sum(m['related_work'] is not None for m in mined.values())
print(f'parsed {len(mined)} papers, {n_rw} with a related-work section; failures: {fail}')

### 4. Resolve cited titles → OpenAlex (abstracts, year, citation count)
Title match is accepted only above a fuzzy-similarity threshold, so a wrong paper is rarely attached. The cache makes re-runs free.

In [ ]:
from rapidfuzz import fuzz

CACHE_PATH = DATA / 'openalex_cache.json'
oa_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}

def rebuild_abstract(inv):
    if not inv: return None
    pos = [(i, w) for w, idxs in inv.items() for i in idxs]
    return ' '.join(w for _, w in sorted(pos))

def resolve_title(title):
    title = (title or '').strip()
    if len(title) < 12: return None
    if title in oa_cache: return oa_cache[title]
    try:
        r = requests.get('https://api.openalex.org/works', timeout=30, params={
            'search': title[:300], 'per_page': 3, 'mailto': OPENALEX_MAILTO,
            'select': 'id,title,publication_year,cited_by_count,abstract_inverted_index,doi'})
        best = None
        for w in r.json().get('results', []):
            score = fuzz.token_sort_ratio(title.lower(), (w.get('title') or '').lower())
            if score >= 88 and (best is None or score > best['match_score']):
                best = {'openalex_id': w['id'], 'title': w['title'], 'year': w.get('publication_year'),
                        'cited_by_count': w.get('cited_by_count'), 'doi': w.get('doi'),
                        'abstract': rebuild_abstract(w.get('abstract_inverted_index')),
                        'match_score': score}
    except Exception:
        return None          # transient error: don't cache, retry next run
    oa_cache[title] = best
    time.sleep(0.11)
    return best

titles = sorted({t for m in mined.values() for t in m['keymap'].values()})
todo = [t for t in titles if t not in oa_cache]
print(f'{len(titles)} unique cited titles, {len(todo)} to resolve')
for i, t in enumerate(tqdm(todo)):
    resolve_title(t)
    if i % 200 == 0: CACHE_PATH.write_text(json.dumps(oa_cache))
CACHE_PATH.write_text(json.dumps(oa_cache))
hit = sum(oa_cache.get(t) is not None for t in titles)
print(f'resolved {hit}/{len(titles)} ({hit/max(1,len(titles)):.0%}); '
      f'{sum(1 for t in titles if (oa_cache.get(t) or {}).get("abstract"))} have abstracts')

### 5. Write the datasets

In [ ]:
def cited_meta(m, key):
    title = m['keymap'].get(key)
    return title, (oa_cache.get(title) if title else None)

# ---- citation_contexts.jsonl : training data for the relation classifier ----
n_ctx = 0
with (DATA / 'citation_contexts.jsonl').open('w') as f:
    for aid, m in mined.items():
        rw_text = (m['related_work'] or {}).get('text', '')
        for c in m['all_contexts']:
            title, oa = cited_meta(m, c['cite_key'])
            if title is None: continue
            f.write(json.dumps({
                'citing_id': aid, 'citing_date': m['published'], 'split': m['split'],
                'cite_key': c['cite_key'], 'cited_title': title,
                'cited_openalex_id': oa and oa['openalex_id'],
                'cited_year': oa and oa['year'],
                'cite_sentence': c['cite_sentence'], 'context': c['context'],
                'n_cites_in_sentence': c['n_cites_in_sentence'],
                'in_related_work': c['cite_sentence'] in rw_text,
            }) + '\n'); n_ctx += 1

# ---- rw_pairs.jsonl : (cited abstracts -> human related-work) benchmark ----
MIN_COVERAGE = 0.6   # fraction of cited papers we have abstracts for
kept, dropped = 0, 0
with (DATA / 'rw_pairs.jsonl').open('w') as f:
    for aid, m in mined.items():
        rw = m['related_work']
        if not rw or len(rw['cited_keys']) < 3: continue
        cited = []
        for k in rw['cited_keys']:
            title, oa = cited_meta(m, k)
            cited.append({'cite_key': k, 'title': title,
                          'abstract': oa and oa['abstract'],
                          'year': oa and oa['year'],
                          'openalex_id': oa and oa['openalex_id']})
        coverage = sum(c['abstract'] is not None for c in cited) / len(cited)
        if coverage < MIN_COVERAGE:
            dropped += 1; continue
        f.write(json.dumps({
            'arxiv_id': aid, 'split': m['split'], 'published': m['published'],
            'title': m['title'], 'abstract': m['abstract'],
            'rw_title': rw['title'], 'rw_text': rw['text'], 'rw_paragraphs': rw['paragraphs'],
            'cited': cited, 'abstract_coverage': round(coverage, 3),
        }) + '\n'); kept += 1

print(f'citation_contexts.jsonl: {n_ctx} rows')
print(f'rw_pairs.jsonl: {kept} papers kept, {dropped} dropped for low abstract coverage')

### 6. Sanity checks
Read a few samples by eye before moving on. Bad parsing here silently poisons every later stage.

In [ ]:
import pandas as pd
ctx = pd.read_json(DATA / 'citation_contexts.jsonl', lines=True)
rw  = pd.read_json(DATA / 'rw_pairs.jsonl', lines=True)

print(ctx.groupby('split').agg(rows=('cite_key', 'size'), papers=('citing_id', 'nunique'),
                               in_rw=('in_related_work', 'mean')).round(2))
rw['n_cited'] = rw['cited'].str.len()
rw['rw_words'] = rw['rw_text'].str.split().str.len()
print(rw.groupby('split')[['n_cited', 'rw_words', 'abstract_coverage']].median())

# leakage check: no test paper should be older than the newest train paper
assert rw[rw.split=='test'].published.min() > rw[rw.split=='train'].published.max(), 'time split overlap!'

for _, r in ctx.sample(3, random_state=1).iterrows():
    print('\n---', r.cited_title, '\n', r.context[:400])

### Next
- **02_relation_labels.ipynb** — define the relation taxonomy, label a few thousand contexts with an LLM, hand-check ~200 for agreement, then fine-tune SciBERT (component A).
- `rw_pairs.jsonl` (test split) is the frozen evaluation benchmark for #6; don't tune anything on it.
